# F6 — Entrenamiento PPO (Agente A y Agente B) en Kaggle

**Escape hatch de cómputo** para los 10 entrenamientos de la Fase 6 (5 seeds × Agente A/B) sin atar tu Mac.

PPO con `MlpPolicy` es **CPU-bound** (redes pequeñas; el cuello de botella es el step del entorno), así que la GPU aporta poco al PPO en sí — pero el CPU de Kaggle es decente y la sesión de 12 h sobra.

Flujo:
1. Setup: clona repo + instala dependencias.
2. Inputs: `scripts/01` + `scripts/02` (features, splits, scalers; ~5-7 min).
3. Dataset sintético de F4 (`run_42`): adjuntar como Kaggle Dataset o regenerar.
4. Smoke test: valida el pipeline F6 end-to-end.
5. Multirun: 5 seeds × Agente A/B (10 corridas, F6-T10/T11).
6. Empaqueta `outputs/ppo_runs/` + `mlruns/` en `.tar.gz` para descarga.

**Tras descargar**: desempaca en el repo local para las Fases 7-8.

## 1. Setup: clonar repo + dependencias

Repo: [Diegbys/tfm-model-unir](https://github.com/Diegbys/tfm-model-unir).

**Pre-requisitos**:
- Settings → Internet: **ON**.
- Accelerator: **GPU opcional** — solo acelera la regeneración de TimeGAN (paso 3); el entrenamiento PPO corre en CPU.

**Dependencias**: Kaggle ya trae `torch`, `numpy`, `pandas`, `scikit-learn`, `pyarrow`, `matplotlib`. `stable-baselines3` reusa el `torch` de Kaggle (no lo reinstala). `gymnasium` se fija a `<1.1` para casar con SB3 ≥ 2.4.

In [ ]:
!git clone https://github.com/Diegbys/tfm-model-unir.git /kaggle/working/model-tfm
%cd /kaggle/working/model-tfm
!pip install -q hydra-core omegaconf mlflow ta yfinance pandas-datareader \
    pandas_market_calendars "gymnasium>=0.29,<1.1" "stable-baselines3>=2.4,<3.0"

In [ ]:
import stable_baselines3 as sb3, gymnasium as gym, torch
print('stable-baselines3:', sb3.__version__)
print('gymnasium:', gym.__version__)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Generar inputs (Fases 1-3)

Reproduce la descarga de datos y el feature engineering en Kaggle (~5-7 min). Deja `data/processed/features.parquet`, `data/splits/` y `models/scalers/`.

In [ ]:
!python scripts/01_download_data.py
!python scripts/02_build_features.py

## 3. Dataset sintético de la Fase 4 (`run_42`)

El Agente B necesita `data/synthetic/run_42/synthetic_dataset.parquet` — un artefacto de la Fase 4 (TimeGAN), **no** regenerable por los scripts 01-02.

Dos opciones:
- **Rápida (recomendada)**: sube los artefactos de F4 (`f4_artifacts.tar.gz`, que genera `02_train_timegan_kaggle.ipynb`) como *Kaggle Dataset*, adjúntalo y descomenta el bloque `KAGGLE_F4` de la celda siguiente.
- **Self-contained**: si el dataset falta, la celda regenera TimeGAN para el seed 42 (~1-2 h en T4).

In [ ]:
import os, shutil

SYNTH = 'data/synthetic/run_42/synthetic_dataset.parquet'

# Si adjuntaste los artefactos de F4 como Kaggle Dataset, ajusta la ruta y
# descomenta este bloque para copiarlos en vez de regenerar:
# KAGGLE_F4 = '/kaggle/input/tfm-f4-artifacts'
# if os.path.isdir(KAGGLE_F4):
#     shutil.copytree(f'{KAGGLE_F4}/data/synthetic', 'data/synthetic', dirs_exist_ok=True)

if os.path.exists(SYNTH):
    print('Dataset sintético presente:', SYNTH)
else:
    print('Falta el dataset sintético — regenerando F4 para seed=42 (~1-2 h en T4)...')
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    !python scripts/03_train_timegan.py seed=42 timegan.device={dev}
    !python scripts/04_evaluate_synthetic.py seed=42 timegan.device={dev}
    assert os.path.exists(SYNTH), 'la regeneración de F4 no produjo el dataset sintético'

## 4. Smoke test

Valida el pipeline F6 end-to-end con timesteps reducidos (`ppo=smoke`) antes de comprometer las 10 corridas completas.

In [ ]:
!python scripts/05_train_agent.py experiment=agent_a ppo=smoke seed=42
!python scripts/05_train_agent.py experiment=agent_b ppo=smoke seed=42

## 5. Multirun completo (F6-T10 / F6-T11)

5 seeds × Agente A/B = 10 corridas de 1M timesteps. Misma config y mismos seeds para A y B; lo único que cambia es el dataset (regla de oro de la comparativa, ADR §Fase 6).

In [ ]:
!python scripts/05_train_agent.py -m experiment=agent_a,agent_b seed=0,1,42,123,1337

## 6. Empaquetado para descarga

El `.tar.gz` aparece en el panel "Output" del notebook. Tras descargar:

```bash
tar xzf ~/Downloads/f6_artifacts.tar.gz -C ~/Documents/Universidad/Q2/TFM/model-tfm/
```

Eso deja `outputs/ppo_runs/` y `mlruns/` listos para las Fases 7-8 (backtesting y análisis estadístico).

In [ ]:
!tar czf /kaggle/working/f6_artifacts.tar.gz outputs/ppo_runs/ mlruns/
!ls -lh /kaggle/working/f6_artifacts.tar.gz